# Custom MT Button-Press + Push Results Notebook

Notebook για το custom multi-task experiment:

```text
button-press-v3 + push-v3
```

Το training script εκπαιδεύει ένα κοινό PPO policy με observation:

```text
Meta-World observation + one-hot task ID
button-press-v3 -> [1, 0]
push-v3         -> [0, 1]
```

Το notebook υποστηρίζει δύο ειδών evaluation results:

1. **Custom wrapper evaluation** από `evaluate_button_push_custom_mt.py`
2. **Individual MT1 env evaluation** από `evaluate_custom_mt_button_push_individual.py`
.

## Load result files

Default folders:

```text
button_push_eval_results/
individual_env_eval_results/
```

In [ ]:
from pathlib import Path
import pandas as pd # type: ignore
import numpy as np # type: ignore
import matplotlib.pyplot as plt # type: ignore

# 1. Results from evaluate_button_push_custom_mt.py
CUSTOM_EVAL_DIR = Path('button_push_eval_results')
CUSTOM_SUMMARY_PATH = CUSTOM_EVAL_DIR / 'button_push_eval_summary.csv'
CUSTOM_RAW_PATH = CUSTOM_EVAL_DIR / 'button_push_eval_raw_episodes.csv'
CUSTOM_PIVOT_PATH = CUSTOM_EVAL_DIR / 'button_push_success_rate_pivot.csv'

# 2. Results from evaluate_custom_mt_button_push_individual.py
INDIV_EVAL_DIR = Path('individual_env_eval_results')
INDIV_FINAL_PATH = INDIV_EVAL_DIR / 'individual_env_final_summary.csv'
INDIV_BY_SEED_PATH = INDIV_EVAL_DIR / 'individual_env_summary_by_seed.csv'
INDIV_RAW_PATH = INDIV_EVAL_DIR / 'individual_env_raw_episodes.csv'
INDIV_PIVOT_PATH = INDIV_EVAL_DIR / 'individual_env_success_pivot.csv'

custom_summary = pd.read_csv(CUSTOM_SUMMARY_PATH) if CUSTOM_SUMMARY_PATH.exists() else None
custom_raw = pd.read_csv(CUSTOM_RAW_PATH) if CUSTOM_RAW_PATH.exists() else None
custom_pivot = pd.read_csv(CUSTOM_PIVOT_PATH) if CUSTOM_PIVOT_PATH.exists() else None

indiv_final = pd.read_csv(INDIV_FINAL_PATH) if INDIV_FINAL_PATH.exists() else None
indiv_by_seed = pd.read_csv(INDIV_BY_SEED_PATH) if INDIV_BY_SEED_PATH.exists() else None
indiv_raw = pd.read_csv(INDIV_RAW_PATH) if INDIV_RAW_PATH.exists() else None
indiv_pivot = pd.read_csv(INDIV_PIVOT_PATH) if INDIV_PIVOT_PATH.exists() else None

print('Custom wrapper summary:', CUSTOM_SUMMARY_PATH, 'exists=', custom_summary is not None)
print('Custom wrapper raw:', CUSTOM_RAW_PATH, 'exists=', custom_raw is not None)
print('Individual env final:', INDIV_FINAL_PATH, 'exists=', indiv_final is not None)
print('Individual env by seed:', INDIV_BY_SEED_PATH, 'exists=', indiv_by_seed is not None)

if custom_summary is not None:
    print('\nCustom wrapper summary shape:', custom_summary.shape)
    display(custom_summary.head())

if indiv_final is not None:
    print('\nIndividual env final shape:', indiv_final.shape)
    display(indiv_final.head())

## 1. Custom wrapper evaluation

In [ ]:
if custom_summary is None:
    print('No custom wrapper summary found. Run evaluate_button_push_custom_mt.py first.')
else:
    display(custom_summary)
    pivot = custom_summary.pivot_table(index='config', columns='task_name', values='success_rate')
    display(pivot)

    configs = list(pivot.index)
    tasks = list(pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, pivot[task], width, label=task)
    ax.set_title('Custom MT button/push success rate by config')
    ax.set_xlabel('PPO config')
    ax.set_ylabel('Success rate')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Custom wrapper first-success step

Lower is better.

In [ ]:
if custom_summary is None:
    print('No custom wrapper summary found.')
else:
    fs = custom_summary.pivot_table(index='config', columns='task_name', values='avg_first_success_step')
    display(fs)

    configs = list(fs.index)
    tasks = list(fs.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, fs[task], width, label=task)
    ax.set_title('Custom MT avg first success step by config')
    ax.set_xlabel('PPO config')
    ax.set_ylabel('Avg first success step')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Custom wrapper per-seed breakdown

Το raw CSV έχει rows ανά episode. Εδώ βλέπουμε αν κάποιο αποτέλεσμα εξαρτάται από συγκεκριμένο eval seed.

In [ ]:
if custom_raw is None:
    print('No custom raw episodes found.')
else:
    display(custom_raw.head())
    by_seed = (
        custom_raw.groupby(['config', 'task_name', 'eval_seed'])
        .agg(
            success_rate=('success', 'mean'),
            avg_return=('return', 'mean'),
            avg_first_success_step=('first_success_step', 'mean'),
            episodes=('success', 'count'),
        )
        .reset_index()
    )
    display(by_seed)

    for task in sorted(by_seed['task_name'].unique()):
        d_task = by_seed[by_seed['task_name'] == task]
        fig, ax = plt.subplots(figsize=(8, 5))
        for config in sorted(d_task['config'].unique()):
            d = d_task[d_task['config'] == config].sort_values('eval_seed')
            ax.plot(d['eval_seed'], d['success_rate'], marker='o', label=config)
        ax.set_title(f'Custom wrapper success across eval seeds — {task}')
        ax.set_xlabel('Eval seed')
        ax.set_ylabel('Success rate')
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

## 4. Individual MT1 env evaluation

Αυτό το evaluation τεστάρει το custom MT policy πάνω στα κανονικά individual MT1 envs, με χειροκίνητο one-hot task ID.

In [ ]:
if indiv_final is None:
    print('No individual env final summary found. Run evaluate_custom_mt_button_push_individual.py first.')
else:
    display(indiv_final)
    pivot = indiv_final.pivot_table(index='config', columns='env_name', values='mean_success_rate')
    display(pivot)

    configs = list(pivot.index)
    envs = list(pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(envs), 1)

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, env_name in enumerate(envs):
        ax.bar(x + (i - (len(envs)-1)/2) * width, pivot[env_name], width, label=env_name)
    ax.set_title('Individual MT1 env success rate by config')
    ax.set_xlabel('PPO config')
    ax.set_ylabel('Mean success rate')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Compare custom-wrapper vs individual-env evaluation

Αν και τα δύο evaluations συμφωνούν, αυτό ενισχύει το συμπέρασμα ότι το policy όντως έμαθε τα tasks και όχι μόνο το custom wrapper.

In [ ]:
if custom_summary is None or indiv_final is None:
    print('Need both custom_summary and indiv_final to compare.')
else:
    cw = custom_summary.rename(columns={
        'config': 'config',
        'task_name': 'task',
        'success_rate': 'custom_wrapper_success',
        'avg_return': 'custom_wrapper_return',
        'avg_first_success_step': 'custom_wrapper_first_success',
    })[['config', 'task', 'custom_wrapper_success', 'custom_wrapper_return', 'custom_wrapper_first_success']]

    indiv = indiv_final.rename(columns={
        'env_name': 'task',
        'mean_success_rate': 'individual_env_success',
        'mean_return': 'individual_env_return',
        'mean_first_success_step': 'individual_env_first_success',
    })[['config', 'task', 'individual_env_success', 'individual_env_return', 'individual_env_first_success']]

    comparison = cw.merge(indiv, on=['config', 'task'], how='outer')
    comparison['success_diff_custom_minus_individual'] = comparison['custom_wrapper_success'] - comparison['individual_env_success']
    display(comparison)

    labels = comparison['config'] + '\n' + comparison['task']
    x = np.arange(len(comparison))
    width = 0.38

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width/2, comparison['custom_wrapper_success'], width, label='custom wrapper')
    ax.bar(x + width/2, comparison['individual_env_success'], width, label='individual env')
    ax.set_title('Custom wrapper vs individual env success')
    ax.set_xlabel('Config / task')
    ax.set_ylabel('Success rate')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Save figures

In [ ]:
fig_dir = Path('button_push_custom_mt_figures')
fig_dir.mkdir(parents=True, exist_ok=True)
saved = []

if custom_summary is not None:
    pivot = custom_summary.pivot_table(index='config', columns='task_name', values='success_rate')
    configs = list(pivot.index)
    tasks = list(pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(tasks), 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, task in enumerate(tasks):
        ax.bar(x + (i - (len(tasks)-1)/2) * width, pivot[task], width, label=task)
    ax.set_title('Custom MT button/push success rate by config')
    ax.set_xlabel('PPO config')
    ax.set_ylabel('Success rate')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    out = fig_dir / 'custom_wrapper_success_by_config.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.close(fig)
    saved.append(out)

if indiv_final is not None:
    pivot = indiv_final.pivot_table(index='config', columns='env_name', values='mean_success_rate')
    configs = list(pivot.index)
    envs = list(pivot.columns)
    x = np.arange(len(configs))
    width = 0.8 / max(len(envs), 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, env_name in enumerate(envs):
        ax.bar(x + (i - (len(envs)-1)/2) * width, pivot[env_name], width, label=env_name)
    ax.set_title('Individual MT1 env success rate by config')
    ax.set_xlabel('PPO config')
    ax.set_ylabel('Mean success rate')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    out = fig_dir / 'individual_env_success_by_config.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.close(fig)
    saved.append(out)

print('Saved figures:')
for p in saved:
    print(' -', p)

## 7. Suggested thesis interpretation

Μπορείς να χρησιμοποιήσεις κάτι σαν αυτό:

> A custom multi-task environment was created by combining `button-press-v3` and `push-v3`. A single PPO policy was trained on both tasks using a one-hot task identifier appended to the Meta-World observation. Evaluation was performed in two ways: first inside the custom two-task wrapper, and second directly on the individual MT1 environments while preserving the correct one-hot task ID. This verifies whether the learned policy transfers from the custom wrapper to the original task environments.

Στα ελληνικά:

> Δημιουργήθηκε ένα custom multi-task περιβάλλον συνδυάζοντας τα `button-press-v3` και `push-v3`. Ένα κοινό PPO policy εκπαιδεύτηκε και στα δύο tasks, με one-hot task identifier στο τέλος του Meta-World observation. Η αξιολόγηση έγινε με δύο τρόπους: πρώτα μέσα στο custom two-task wrapper και μετά απευθείας στα individual MT1 environments, διατηρώντας το σωστό one-hot task ID. Έτσι ελέγχεται αν το policy που έμαθε στο custom wrapper μεταφέρεται σωστά στα αρχικά task environments.